# Nemotron 3.5 Parse — document layout on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Nemotron-3.5-Parse-layout.ipynb)

[`nvidia/Nemotron-3.5-Parse`](https://huggingface.co/nvidia/Nemotron-3.5-Parse) is NVIDIA's 0.63B document parser (OpenMDW-1.1): a C-RADIOv4-SO400M vision encoder plus a 10-layer mBART decoder. It returns semantic classes, content-normalized bounding boxes, and page text in **reading order**. The default prompt asks for boxes, classes, and markdown, including text inside figures. This notebook is the **layout** companion to the OCR notebooks ([LightOnOCR-3](LightOnOCR-3-0.8B-document-ocr.ipynb), OvisOCR3): boxes and reading-order Markdown/HTML, not a full OCR bake-off.

**Licence:** [OpenMDW-1.1](https://openmdw.ai/license/1-1/). The bundled tokenizer is MIT; see the [tokenizer attribution](https://huggingface.co/nvidia/Nemotron-3.5-Parse/blob/main/TOKENIZER_ATTRIBUTION.md).

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM, **no native bfloat16**, and **no FlashAttention 2**. The card's Hopper path is BF16 + vLLM/FA3 with an MTP head; this notebook stays on the **Transformers** path, **float16**, and PyTorch **SDPA** (the custom decoder already calls `scaled_dot_product_attention`). Hub weights are about 2.5 GB on disk; float16 in VRAM is about **1.3 GB**. A 2048×1664 page plus a few thousand generated tokens should stay well under 16 GB. That estimate is from the parameter count and the card's max canvas, not a measured Colab run.

**Not in this notebook:** vLLM, FlashAttention 3, the bundled one-token MTP head, or the card's double/triple-pass crop refinement (`DocumentParser`). Those need the Nemotron vLLM plugin.

**Links:** [model card](https://huggingface.co/nvidia/Nemotron-3.5-Parse) · [OpenMDW-1.1](https://openmdw.ai/license/1-1/) · [Transformers example](https://huggingface.co/nvidia/Nemotron-3.5-Parse#usage-example)


In [ ]:
%pip install -q "transformers==5.20.1" "timm==1.0.22" "beautifulsoup4==4.13.3" "pypdfium2==5.14.0" pillow torchvision

# The card pins transformers==5.14.1, torchvision==0.26.0, and timm==1.0.22.
# 5.20.1 matches the other notebooks in this repo and sits above that floor.
# Custom model + processor code is loaded with trust_remote_code (no Nemotron pip package).
# timm supplies the ViT Block used by the shipped C-RADIO encoder.
# beautifulsoup4 is required by the card's postprocessing helpers (tables → HTML).
# pypdfium2 renders PDF pages. pillow and torchvision are already on Colab.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.14.1"), (
    f"transformers {transformers.__version__} is loaded; Nemotron 3.5 Parse needs 5.14.1+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2 and no native bfloat16.
# Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# Forced float16: do not follow the card's torch.bfloat16 Hopper default.
DTYPE = torch.float16
major, minor = torch.cuda.get_device_capability()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
if major < 8:
    print("T4-class GPU: float16 + SDPA (no BF16, no FlashAttention 2).")
else:
    print("This GPU can run BF16; this notebook still loads float16 to match the T4 path.")


## 1. Load Nemotron 3.5 Parse

`AutoModel` / `AutoProcessor` with `trust_remote_code=True`, as on the model card. The first run downloads about 2.5 GB. Generation is greedy with `repetition_penalty=1.1` and the card's default prompt (`boxes + classes + markdown + text-in-picture`).


In [ ]:
from transformers import AutoModel, AutoProcessor, GenerationConfig

MODEL_ID = "nvidia/Nemotron-3.5-Parse"
TASK_PROMPT = "</s><s><predict_bbox><predict_classes><output_markdown><predict_text_in_pic>"

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype=DTYPE,
).to("cuda").eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
generation_config = GenerationConfig.from_pretrained(MODEL_ID)
generation_config.do_sample = False
generation_config.num_beams = 1
generation_config.repetition_penalty = 1.1

print("dtype:", model.dtype)
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


The card's postprocessing helpers turn the raw serialization into reading-order Markdown (HTML tables keep merged cells) and pixel boxes. They are fetched from the Hub into the Hugging Face cache and put on `sys.path`. There is no `nemotron-parse` package install and no vLLM extra.


In [ ]:
import sys
from pathlib import Path

from huggingface_hub import snapshot_download

HELPERS = Path(snapshot_download(
    MODEL_ID,
    allow_patterns=["postprocessing.py", "latex2html.py", "nemotron_parse/**"],
    ignore_patterns=["nemotron_parse/inference.py", "nemotron_parse/batch.py"],
))
if str(HELPERS) not in sys.path:
    sys.path.insert(0, str(HELPERS))

from postprocessing import postprocess_document

print("helpers:", HELPERS)


## 2. PDF page and a page image

Samples:

* page 1 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (arXiv:1706.03762) — title, authors, abstract, two-column body (reading order)
* page 8 of the same PDF — Table 2 (Markdown with HTML tables)
* the SROIE receipt used in the LightOnOCR notebook — a photographed page

PDFs are rendered at 400 DPI and limited to a 2048 px longest edge, then the processor packs them onto the card's 2048×1664 NaViT canvas. Coordinates are **content-normalized** (scale x by image width and y by height). `max_new_tokens=4096` is a T4-friendly cap; the card default is 8744.


In [ ]:
import io
import time

import requests
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFont

PAPER_URL = "https://arxiv.org/pdf/1706.03762"
RECEIPT_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
CLASS_COLORS = {
    "Title": (220, 40, 40),
    "Section-header": (200, 90, 20),
    "Text": (30, 90, 200),
    "List-item": (20, 140, 160),
    "Table": (140, 40, 180),
    "Chart": (180, 40, 120),
    "Picture": (40, 150, 70),
    "Caption": (90, 90, 90),
    "Formula": (180, 120, 20),
    "Footnote": (110, 110, 110),
    "Page-header": (150, 150, 150),
    "Page-footer": (150, 150, 150),
}


def fetch_bytes(url, attempts=4):
    for attempt in range(attempts):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
            response.raise_for_status()
            return response.content
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))


def fetch_image(url):
    return Image.open(io.BytesIO(fetch_bytes(url))).convert("RGB")


def render_pdf_page(url, page_index, longest=2048):
    import pypdfium2 as pdfium

    pdf = pdfium.PdfDocument(fetch_bytes(url))
    image = pdf[page_index].render(scale=400 / 72).to_pil().convert("RGB")
    image.thumbnail((longest, longest))
    return image


def limit_edge(image, longest=2048):
    image = image.copy()
    image.thumbnail((longest, longest))
    return image


def parse_page(image, max_new_tokens=4096):
    inputs = processor(
        images=[image],
        text=TASK_PROMPT,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)
    generation_config.max_new_tokens = max_new_tokens
    with torch.inference_mode():
        output_ids = model.generate(**inputs, generation_config=generation_config)
    raw = processor.batch_decode(output_ids, skip_special_tokens=False)[0]
    document = postprocess_document(raw, image_size=image.size)
    if torch.cuda.is_available():
        print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
    return document


def draw_layout(image, document):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 16)
    except OSError:
        font = ImageFont.load_default()
    for index, block in enumerate(document["blocks"], start=1):
        box = block.get("bbox_pixels")
        if not box:
            continue
        color = CLASS_COLORS.get(block["class"], (200, 80, 40))
        x1, y1, x2, y2 = box
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        label = f"{index} {block['class']}"
        tx, ty = x1 + 2, max(0, y1 - 16)
        draw.rectangle((tx, ty, tx + 8 * len(label), ty + 16), fill=color)
        draw.text((tx + 2, ty), label, fill=(255, 255, 255), font=font)
    return canvas


def show_document(image, document):
    boxed = draw_layout(image, document)
    print(f"{len(document['blocks'])} blocks in reading order")
    for index, block in enumerate(document["blocks"], start=1):
        preview = " ".join(block["text"].split())[:80]
        print(f"{index:3}  {block['class']:<16}  {preview}")
    if document.get("warnings"):
        print("warnings:", document["warnings"])
    display(boxed)
    display(Markdown(document["markdown"] or "_(empty markdown)_"))
    return boxed


paper_title = render_pdf_page(PAPER_URL, page_index=0)
paper_table = render_pdf_page(PAPER_URL, page_index=7)  # printed page 8, Table 2
receipt = limit_edge(fetch_image(RECEIPT_URL))
print("title page", paper_title.size, "| table page", paper_table.size, "| receipt", receipt.size)


### Title page — reading-order boxes and Markdown

Page 1 of the paper. Boxes are numbered in the order the model emitted them. The Markdown below is the same sequence, with titles and section headers promoted by the card's helpers.


In [ ]:
display(paper_title)
title_doc = parse_page(paper_title)
show_document(paper_title, title_doc)


### Table page — HTML tables in reading order

Page 8 holds Table 2. Table blocks are converted to HTML so merged cells survive; the Markdown view below embeds those tables.


In [ ]:
display(paper_table)
table_doc = parse_page(paper_table)
show_document(paper_table, table_doc)


### Receipt — a photographed page

Same layout path on a page image rather than a rendered PDF. This is still boxes + reading order, not a contest with the [LightOnOCR-3](LightOnOCR-3-0.8B-document-ocr.ipynb) grounding notebook.


In [ ]:
display(receipt)
receipt_doc = parse_page(receipt)
show_document(receipt, receipt_doc)


## 3. Caveats

* **OpenMDW-1.1**, not Apache-2.0. Review [the licence](https://openmdw.ai/license/1-1/) before production use. The tokenizer is MIT.
* **Transformers only.** The card's faster path is vLLM + FA3 + the bundled MTP head on Hopper. This notebook does not install that plugin and does not use FlashAttention-2.
* **No BF16.** The card example loads `torch.bfloat16`. A T4 cannot do that natively, so weights stay in **float16**.
* **Single pass.** Double/triple-pass table and chart crops live on `nemotron_parse.inference.DocumentParser` (vLLM). The Transformers demo is one full-page generate.
* **Layout complement, not an OCR bake-off.** Use this notebook for semantic boxes, classes, and reading-order Markdown/HTML. For transcription-first OCR (receipts, handwriting, grounded blocks), use the LightOnOCR-3 notebook in this repo, or OvisOCR3. Nemotron 3.5 Parse can read text, but that is not the comparison this notebook is making.
* **Remote code + timm.** `trust_remote_code=True` executes the repo's processor and model files. `timm` must be installed before the first `from_pretrained`.
* **Token budget.** Dense pages can hit `max_new_tokens=4096`. Raise it toward the card's 8744 on a longer session if a page truncates.
* **Input size.** Recommended maximum is 1664×2048. Larger pages are packed onto that canvas.
